# Lesson 61 · `nn.Module` in practice — giving a million parameters a home address: parameter registration, `Sequential`, and saving models

**Goal**: rebuild L57's Linear→ReLU→Linear with `nn.Module`. First get the chain of registering, saving and loading parameters straight, then take a look at the syntactic sugar of `Sequential` along the way.

> **Principle**: being able to use `nn` still means understanding what's underneath (L54–L58); the API just wraps the underlying maths — it's not an excuse for a black box.

🔗 **Aurora connection**: the Month 2 keyword-spotting model (KWS), Month 3 ASR fine-tuning and the Month 5 LLM all use `nn.Module` as their skeleton. Master its registration mechanism and its serialization interface, and training, saving and loading later on all go through one consistent entry point.

← **Previous lesson** · [L60 · How autograd works](L60_pytorch_autograd.ipynb)

> Last lesson covered **how autograd works**: the `grad_fn` computation graph, `backward()`, `retain_grad` and gradient accumulation, checked numerically against the hand-written `Value` engine.
> This lesson explores **`nn.Module` in practice**.

## This lesson's story: where the parameters "live" decides whether the model can be saved and loaded

L57's `MLP` class can run forward and backward — but it has a hidden weakness: **its parameters are `Value` objects scattered across Python variables**. Once the model grows, it's like having parts strewn all over the workbench: to save them to disk you have to pick them up one by one yourself; to move them to the GPU you have to convert them one at a time.

`nn.Module` solves this problem. Think of it as a "parameter registry", a town hall where every parameter gets its address on file:

```python
class MyLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(4, 4))  # ← registered as a parameter

    def forward(self, x):
        return x @ self.weight
```

As long as it's wrapped in `nn.Parameter`, `self.weight` automatically shows up in `model.parameters()`, gets found by the optimizer, gets included in `state_dict()`, and moves to the GPU in one go with `model.to('cuda')`.

**Three key tools**:

| Tool | What it does |
|---|---|
| `nn.Parameter(tensor)` | registers a trainable parameter |
| `model.parameters()` | an iterator over all parameters |
| `model.state_dict()` | a dict, for saving/loading model state |

**Aurora connection**: `aurora.llm.KVCache` is currently a pure-numpy inference component (the from-scratch principle; it needs no gradients); the planned training-side `src/aurora/ml/` module will be built entirely on `nn.Module`. And the key/value cache inside KVCache — "takes no part in gradients, but has to move devices along with the model" — corresponds in the torch world exactly to `register_buffer()`, which this lesson covers.

This lesson's task: implement `AudioMLP(nn.Module)` — two linear layers + ReLU — and use the parameter total of 3,274 to check whether you've really absorbed the registration rules.

In [ ]:
import torch
import torch.nn as nn

## 1. `forward()` and automatic parameter registration

`nn.Module` requires subclasses to implement `forward()`, and you call it as `model(x)` rather than `model.forward(x)` —
the former triggers hooks, the latter bypasses them.

There are two ways parameters get registered:
- assign a built-in layer (`nn.Linear`, `nn.Conv1d` …): its parameters are registered automatically
- assign a bare tensor: you **must** wrap it with `nn.Parameter(tensor)`, or it won't be tracked

```python
# right: the parameter is registered
self.w = nn.Parameter(torch.randn(10, 5))
# wrong: not tracked, the optimizer can't see it
self.w = torch.randn(10, 5)
```

### Deep dive: why is Linear's weight `(out_features, in_features)` rather than `(in_features, out_features)`?

Plenty of beginners get confused here: `nn.Linear(40, 64)` takes 40-dimensional input and gives 64-dimensional output, so the weight should be `(40, 64)`, right? But PyTorch stores `(64, 40)` — why the other way round?

**The mathematical truth**: the formula a Linear layer actually computes is
$$y = x \cdot W^T + b$$

where:
- $x$ is the input, shape `(batch, in_features)` = `(batch, 40)`
- $W$ is stored in memory with shape `(out_features, in_features)` = `(64, 40)`
- $W^T$ (the transpose) has shape `(in_features, out_features)` = `(40, 64)`
- $y$ is the output, shape `(batch, out_features)` = `(batch, 64)`

**A worked example** (with batch=2, in=40, out=64):

```
input x:              (2, 40)  —— 2 samples, each a 40-dim vector
weight W (stored):    (64, 40)  —— the shape in PyTorch's memory
weight W^T (transp.): (40, 64)  —— the shape actually used in the multiply

matrix multiplication:
  x @ W^T = (2, 40) @ (40, 64) = (2, 64)
            ↑      ↑  ↑       ↑   ↑      ↑
          batch  in  in     out  batch out
```

Why it's designed this way:
1. **Memory-efficient**: the weight matrix is stored by rows (each row holds the weights of one output neuron), so access is sequential and cache-friendly.
2. **Efficient gradient computation**: during backpropagation the gradients can also use efficient batched operations.
3. **Consistent parameter initialization**: fan-in/fan-out initialization rules are read row by row.

**A note against intuition**: you might think, "The input is 40-dim and the output is 64-dim, so a '40→64 transformation' ought to be a 40×64 matrix." — Right, logically it is. But for computational efficiency PyTorch **stores the matrix transposed**, so in memory it looks like 64×40. This gap between the "logical shape" and the "storage shape" is very common.

In [ ]:
# Demo: nn.Parameter registration
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)           # weight + bias registered automatically
        self.scale = nn.Parameter(torch.ones(1))  # registered by hand
        self.buf = torch.zeros(1)            # not registered, just a plain tensor

    def forward(self, x):
        return self.fc(x) * self.scale

tiny = TinyNet()
param_names = [n for n, _ in tiny.named_parameters()]
print('registered parameters:', param_names)  # fc.weight, fc.bias, scale
print('buf does not appear:', 'buf' not in param_names)

### Technical detail: how nn.Module "tracks" submodule attributes

What happens when you write `self.fc = nn.Linear(4, 2)`?

`nn.Module` overrides Python's `__setattr__()` method. Every time you assign an object to `self`, it checks automatically:
- If it's a subclass of `nn.Module` (such as `nn.Linear`), it becomes a "submodule" automatically, and its parameters are tracked.
- If it's an `nn.Parameter`, it becomes a "parameter" automatically.
- If it's a `torch.Tensor`, it isn't tracked (unless you register it explicitly with `register_buffer()`).
- Any other Python object (int, str, list, …) isn't tracked either.

```python
# these are tracked automatically:
self.fc = nn.Linear(4, 2)         # submodule → parameters() finds fc.weight, fc.bias
self.scale = nn.Parameter(...)    # Parameter → parameters() finds it
self.register_buffer('mean', ...) # buffer → state_dict() finds it, but it's not in parameters()

# these are NOT tracked:
self.buf = torch.zeros(1)         # plain Tensor → nobody manages it
self.count = 0                    # plain Python int → nobody manages it
```

**The consequence**: only tracked objects can be:
- found by `model.parameters()` → so the optimizer can update them
- saved by `model.state_dict()` → so model checkpoints include them
- moved by `model.to(device)` → so device transfers apply to them

In [ ]:
# Verify the matrix multiply: how the weight shape matches the computation
import torch
import torch.nn as nn

linear_layer = nn.Linear(40, 64)
print(f"weight shape of Linear(40, 64): {linear_layer.weight.shape}")  # (64, 40)
print(f"bias shape of Linear(40, 64):   {linear_layer.bias.shape}")    # (64,)

# verify the matrix multiply by hand
x = torch.randn(2, 40)  # 2 samples, 40 dims each
W = linear_layer.weight  # shape (64, 40)
b = linear_layer.bias    # shape (64,)

# what PyTorch actually computes inside: y = x @ W.T + b
y_manual = x @ W.T + b  # (2, 40) @ (40, 64) + (64,) = (2, 64)
y_actual = linear_layer(x)  # computed directly with the Linear layer

print(f"\ninput x shape: {x.shape}")
print(f"output y shape: {y_actual.shape}")
print(f"hand computation matches the Linear layer: {torch.allclose(y_manual, y_actual)}")

# explanation: the shape of W.T
print(f"\nshape of W.T (after transposing): {W.T.shape}")  # (40, 64)
print(f"matrix multiply (2, 40) @ (40, 64) gives (2, 64) ✓")

### `register_buffer()` — the right way to register non-parameter state

The `buf` counterexample showed what happens when you **don't register**; the right way is `register_buffer()`. It's meant for a tensor that **needs no gradient, but has to move devices with the model (`.to(device)`) and has to appear in `state_dict()`**.

**A real-world example**:

Suppose your model needs to "remember how many samples it has seen" (for a moving average or a learning-rate schedule). That counter:
- needs no gradient (it takes no part in backpropagation)
- must be saved when the model is saved/loaded (it's part of the checkpoint)
- must follow the model across devices (CPU ↔ GPU)

That's exactly what `register_buffer()` is for:

```python
class ModelWithCounter(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)
        # register the counter with register_buffer: it's in state_dict, but has no gradient
        self.register_buffer('sample_count', torch.tensor(0, dtype=torch.long))

    def forward(self, x):
        self.sample_count += x.shape[0]  # add up the number of samples
        return self.linear(x)

model = ModelWithCounter()
print(model.state_dict())  # sample_count appears here
print(list(model.parameters()))  # sample_count does not appear here (no gradient)
```

> The differences at a glance:
>
> | Method | `parameters()` | `state_dict()` | Gradient | Used for |
> |---|---|---|---|---|
> | `nn.Parameter` | ✅ | ✅ | ✅ | trainable weights |
> | `register_buffer()` | ❌ | ✅ | ❌ | model state (counters, running mean, etc.) |
> | bare tensor attribute | ❌ | ❌ | ❌ | temporaries, not saved |

In [ ]:
# Demo: register_buffer — a model counter in practice
class TrainedSampleCounter(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)
        self.scale = nn.Parameter(torch.ones(1))

        # Option 1: register_buffer — the counter is saved, but has no gradient
        self.register_buffer('total_samples', torch.tensor(0, dtype=torch.long))

        # Option 2: bare tensor (the counterexample) — this one won't be saved
        self.unreg_count = torch.tensor(0, dtype=torch.long)

    def forward(self, x):
        # add up the number of samples
        self.total_samples += x.shape[0]
        self.unreg_count += x.shape[0]
        return self.fc(x) * self.scale

model_counter = TrainedSampleCounter()

# run it a few times and the counter goes up
x_test = torch.randn(3, 4)
model_counter(x_test)
model_counter(x_test)
print(f"after 2 forward passes (3 samples each), total_samples = {model_counter.total_samples.item()}")

# look at the state_dict
state = model_counter.state_dict()
print(f"\nwhat's in the state_dict:")
for key in state.keys():
    print(f"  {key}")

print(f"\ntotal_samples in state_dict? {'total_samples' in state}")
print(f"unreg_count in state_dict? {'unreg_count' in state}")

# and in the parameter iterator?
param_names = [n for n, _ in model_counter.named_parameters()]
print(f"\ntotal_samples in parameters()? {'total_samples' in param_names}")

print("\n✅ register_buffer verified:")
print("   - total_samples is in state_dict (can be saved/loaded)")
print("   - total_samples is not in parameters() (no gradient)")
print("   - unreg_count is not tracked (lost)")

## 2. `named_parameters()` and `state_dict()`

### What is a "submodule"?

When you write `self.fc = nn.Linear(4, 2)`, the `nn.Linear` object is itself an `nn.Module`, so it automatically becomes a **submodule** of the current module. A submodule has its own parameters (weight, bias), and these are discovered "recursively" and automatically.

```python
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 5)     # ← submodule 1 (Linear)
        self.fc2 = nn.Linear(5, 2)      # ← submodule 2 (Linear)
        self.param = nn.Parameter(...)  # ← a direct parameter (not a submodule)

# recursive traversal:
# finds fc1.weight, fc1.bias (from submodule fc1)
# finds fc2.weight, fc2.bias (from submodule fc2)
# finds param (direct parameter)
```

**Why "recursive"?** Because submodules can contain submodules of their own:

```python
class DeepModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(      # ← a submodule (Sequential)
            nn.Linear(20, 10),              # ← submodules inside the Sequential
            nn.ReLU(),
            nn.Linear(10, 5),
        )
        self.classifier = nn.Linear(5, 2)  # ← another submodule

# recursive traversal finds parameters at every level: the Linears inside encoder, classifier's parameters, etc.
```

### `named_parameters()` and `state_dict()`

`named_parameters()` walks all submodules recursively and returns `(name, tensor)` pairs; it's commonly used to count parameters or to set per-layer learning rates.

```python
total = sum(p.numel() for p in model.parameters())
```

`state_dict()` returns an `OrderedDict` whose keys are parameter names and whose values are tensors — this is the standard interface for saving/loading checkpoints (ckpt):

```python
torch.save(model.state_dict(), 'ckpt.pt')
model.load_state_dict(torch.load('ckpt.pt'))
```

When two models have the same structure, you can pass the `state_dict` straight from one to the other to copy the weights.

In [ ]:
# Demo: state_dict key names and shapes
for name, param in tiny.named_parameters():
    print(f'{name:20s}  shape={list(param.shape)}  numel={param.numel()}')

print('\nstate_dict keys:', list(tiny.state_dict().keys()))

### A common misconception about state_dict(): only numbers, no structure

`state_dict()` saves only the weight values; it does **not** save any information about "what kind of model this is". It looks like this:

```python
{
    'fc.weight': tensor([...]),  # just numbers
    'fc.bias': tensor([...]),
    'scale': tensor([...]),
}
```

**The right steps for loading**:
1. **First create an empty model object** (define the architecture)
2. **Then load the weights from the file**

```python
# ❌ wrong: you can't reconstruct the model architecture from a state_dict
loaded_dict = torch.load('model.pt')
# this is just a dict; it has no idea how to be used

# ✅ right: define the model first, then load the weights
model = AudioMLP(40, 64, 10)  # define the architecture
model.load_state_dict(torch.load('model.pt'))  # load the saved weights
```

**Why this design?** Because PyTorch wants saved files to be as small and portable as possible. The model's architecture should be defined explicitly in code; the checkpoint is only responsible for storing parameter values.

## 3. `nn.Sequential`: skip writing `forward()`

`nn.Sequential` chains submodules in order and implements `forward(x) = layer_n(... layer_1(x))` for you, no hand-written loop needed. Submodules are named by integer index (`0`, `1`, `2` …), or you can give them string names with an `OrderedDict`.

```python
net = nn.Sequential(
    nn.Linear(40, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
# equivalent to: x -> Linear -> ReLU -> Linear -> output
```

**When it fits and when it doesn't**:

- **✅ Fits**: each layer's output goes straight into the next layer's input, and the data flows along a single chain — like the example above.
- **❌ Doesn't fit**:
  - **Branches** (one layer's output flows into several next layers):
    ```python
    # can't use Sequential (branch example)
    x -> Conv -> [Branch A (Conv->ReLU) and Branch B (Conv->ReLU)] -> Concat -> Output
    ```
  - **Skip connections** (a layer's input includes not just the previous layer's output but also the output of an earlier layer):
    ```python
    # can't use Sequential (skip-connection example, ResNet style)
    x -> Linear1 -> ReLU -> Linear2 ⊕ Linear1(x) -> Output
                                     ↑ (skip connection: an earlier layer's output is added back in)
    ```

In these more complex cases you have to write `forward()` yourself to control the data flow explicitly.

In [ ]:
# Demo: an nn.Sequential forward pass
seq = nn.Sequential(
    nn.Linear(40, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
x_demo = torch.randn(8, 40)
print('Sequential output shape:', seq(x_demo).shape)  # (8, 10)
print('list of submodules:', list(seq.children()))

In [ ]:
# Demo: Sequential vs a hand-written forward — the two are exactly equivalent
# Option 1: Sequential (recommended, concise code)
class MLPv1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(40, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.layers(x)

# Option 2: hand-written forward (explicit control, more code)
class MLPv2(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(40, 64)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.l1(x)
        x = self.relu(x)
        x = self.l2(x)
        return x

# check that they're equivalent
mlp1 = MLPv1()
mlp2 = MLPv2()

# copy the weights so the parameters are identical
mlp2.l1.weight.data = mlp1.layers[0].weight.data.clone()
mlp2.l1.bias.data = mlp1.layers[0].bias.data.clone()
mlp2.l2.weight.data = mlp1.layers[2].weight.data.clone()
mlp2.l2.bias.data = mlp1.layers[2].bias.data.clone()

x_test = torch.randn(8, 40)
out1 = mlp1(x_test)
out2 = mlp2(x_test)

print(f"Sequential version output shape: {out1.shape}")
print(f"hand-written forward output shape: {out2.shape}")
print(f"do the two versions give the same output: {torch.allclose(out1, out2)}")

print("\n💡 Which to choose:")
print("   - simple structure, no branches/skip connections → Sequential (less code, easy to read)")
print("   - branches, skip connections or conditional paths → hand-written forward (clear, flexible)")

## 4. ✏️ Implement `class AudioMLP(nn.Module)`

**Architecture**: `Linear(in→hidden) → ReLU → Linear(hidden→out)`

**Template**:
```python
class AudioMLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()  # ← required! initializes nn.Module's parameter-tracking system
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_features),
        )

    def forward(self, x):
        return self.layers(x)
```

### Why is `super().__init__()` mandatory?

`super().__init__()` calls the `__init__` method of the parent class `nn.Module`, which does this essential work:
- sets up the parameter-tracking system (the `__setattr__` hook)
- creates internal dicts such as `_parameters`, `_buffers`, `_modules`
- gets everything ready for methods like `parameters()` and `state_dict()`

**What happens if you forget that line?**
```python
# ❌ wrong: forgot super().__init__()
class BadMLP(nn.Module):
    def __init__(self):
        # this line is missing
        self.fc = nn.Linear(40, 64)  # the parameters can't be tracked!

    def forward(self, x):
        return self.fc(x)

bad = BadMLP()  # in fact PyTorch stops right here: AttributeError (cannot assign module before Module.__init__() call)
print(len(list(bad.parameters())))  # never reached — the weights never get registered
```

So `super().__init__()` must be the first line of `__init__` (or at least come before you assign any submodule or parameter).

**Why Sequential**: it saves you writing the chain of calls in `forward()` — the order of the modules is the order of computation.

**Acceptance criteria**:

| Check | Expected value |
|---|---|
| `m(torch.randn(8,40)).shape` | `(8, 10)` |
| `sum(p.numel() for p in m.parameters())` | `3274` |
| `m.layers[0].weight.shape` | `(64, 40)` |

In [ ]:
class AudioMLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        # ✏️ TODO: define self.layers = nn.Sequential(...)
        raise NotImplementedError("TODO: in __init__, define self.layers = nn.Sequential(nn.Linear(in_features, hidden), nn.ReLU(), nn.Linear(hidden, out_features))")

    def forward(self, x):
        # ✏️ TODO: return self.layers(x)
        raise NotImplementedError("TODO: in forward, return self.layers(x)")



In [ ]:
try:
    m = AudioMLP(40, 64, 10)
    out = m(torch.randn(8, 40))
    assert out.shape == (8, 10), f'expected (8,10), got {out.shape}'
    print('✅ AudioMLP forward output shape is correct:', out.shape)
except (NotImplementedError, TypeError) as e:
    print(f'⚠️ Exercise not done yet — implement AudioMLP first: {e}')

## 5. Parameter experiment: check the parameter total

To count the parameters of `AudioMLP(40, 64, 10)` by hand, recall the matrix-multiply formula from earlier, $y = x \cdot W^T + b$:

**Layer 1: Linear(40→64)**
- input dimension: 40, output dimension: 64
- storage shape of the weight $W$: `(64, 40)` (output × input)
- number of weight parameters: $64 \times 40 = 2560$
- bias shape: `(64,)`
- number of bias parameters: $64$
- layer total: $2560 + 64 = 2624$

**Layer 2: Linear(64→10)**
- input dimension: 64, output dimension: 10
- weight shape: `(10, 64)`
- number of weight parameters: $10 \times 64 = 640$
- number of bias parameters: $10$
- layer total: $640 + 10 = 650$

**Total number of parameters**: $2624 + 650 = 3274$

**Why multiply rather than add?** Because every output neuron needs a complete weight vector to process the input. With 64 output neurons, each needing 40 weights, the total is $64 \times 40$.

### A note on initialization: `torch.randn()` vs `torch.zeros()`

- **`torch.randn(shape)`**: samples at random from the standard normal distribution $\mathcal{N}(0, 1)$
  - use: initializing **trainable parameters** (weights, biases)
  - reason: random initialization breaks the symmetry, so neurons learn different features

- **`torch.zeros(shape)`**: an all-zero tensor
  - use: initializing **non-parameter state** (counters, caches, etc.)
  - reason: these variables take no part in gradient computation, so the initial value usually doesn't matter much

`nn.Linear` already initializes its weights randomly inside (strictly speaking with a Kaiming-uniform scheme, a uniform distribution rather than `randn`), so we don't need to call `randn` ourselves.

In [ ]:
try:
    m = AudioMLP(40, 64, 10)
    total = sum(p.numel() for p in m.parameters())
    print(f'parameter total: {total}')
    assert total == 3274, f'expected 3274, got {total}'
    print('✅ parameter total matches the hand count')

    # per-layer parameter breakdown
    for name, p in m.named_parameters():
        print(f'  {name:25s}  {list(p.shape)}  = {p.numel()}')

    # experiment: parameter count with hidden=128
    m128 = AudioMLP(40, 128, 10)
    print(f'\nparameter total with hidden=128: {sum(p.numel() for p in m128.parameters())}')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️ Exercise not done yet — implement AudioMLP first: {e}')

## Wrap-up

`AudioMLP.forward()` chains the linear maps and the ReLU together through `nn.Sequential` and outputs shape `(batch, out_features)`. Mathematically it's the same as L57's hand-written MLP; the only difference is that the lifecycle of the parameters is now managed in one place by `nn.Module`.
`named_parameters()` and `state_dict()` are the direct foundation for saving checkpoints in Aurora's Month 2 KWS and loading pretrained weights in Month 3 ASR.
The next lesson (L62) wraps the feature extractor in a `torch.utils.data.Dataset`, continuing to connect the chain of "how data gets into the model".

## ✏️ Whiteboard challenge: work through nn.Module parameter management by hand (target: 10 minutes)

Cover the screen and derive on paper:

**Question 1**: How many parameters does `AudioMLP(40, 64, 10)` have in total? List them layer by layer.
(L1: 40×64+64=2624, L2: 64×10+10=650, total 2624+650=3274)

**Question 2**: What's the difference between assigning an `nn.Parameter` and a plain `torch.Tensor` to `self.x`?
(A Parameter shows up in parameters() automatically; a plain Tensor doesn't, and isn't updated by the optimizer)

**Question 3**: What's the difference between `model.state_dict()` and `model.parameters()`?
(state_dict contains all buffers and parameters, as a dict; parameters() returns only the Parameters, as an iterator)

**Question 4**: Why does `model.to('cuda')` only work on registered `Parameter`s and `buffer`s?
(The Module only knows about tensors registered through register_parameter/register_buffer (or assigned as a Parameter/submodule); stray variables are outside its jurisdiction)

**Question 5**: What code is the `forward(x)` of `nn.Sequential([Linear, ReLU, Linear])` equivalent to?
(Equivalent to x1=Linear1(x); x2=ReLU(x1); out=Linear2(x2); return out — calling each submodule in order)

When you've finished the derivations, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import torch, torch.nn as nn

# Q1: parameter total
try:
    m_q = AudioMLP(40, 64, 10)
    total_q = sum(p.numel() for p in m_q.parameters())
    assert total_q == 3274, f"parameter total={total_q}"
    l1 = m_q.layers[0].weight.numel() + m_q.layers[0].bias.numel()
    l2 = m_q.layers[2].weight.numel() + m_q.layers[2].bias.numel()
    print(f"Q1 ✅  AudioMLP(40,64,10) parameters: L1={l1}(40×64+64), L2={l2}(64×10+10), total {total_q}")
except (NotImplementedError, TypeError):
    print(f"Q1 ⬜  parameter total = 40×64+64+64×10+10 = 2624+650 = 3274 (checked once AudioMLP is implemented)")

# Q2: Parameter vs plain Tensor
class _TestMod(nn.Module):
    def __init__(self):
        super().__init__()
        self.p = nn.Parameter(torch.zeros(3))
        self.t = torch.zeros(3)  # not registered
    def forward(self, x): return x
tm = _TestMod()
param_names = [n for n, _ in tm.named_parameters()]
assert 'p' in param_names and 't' not in param_names
print(f"Q2 ✅  the Parameter is registered in parameters(), the plain Tensor is not: {param_names}")

# Q3: state_dict vs parameters
class _TestMod2(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.ones(2))
        self.register_buffer('buf', torch.zeros(2))
    def forward(self, x): return x
tm2 = _TestMod2()
sd_keys = list(tm2.state_dict().keys())
param_keys = [n for n, _ in tm2.named_parameters()]
assert 'buf' in sd_keys and 'buf' not in param_keys
print(f"Q3 ✅  state_dict={sd_keys} (includes the buffer), parameters={param_keys} (Parameters only)")

# Q4: moving devices with to() (tests CPU→CPU only, no CUDA needed)
try:
    m_q2 = AudioMLP(4, 8, 2)
    m_q2_cpu = m_q2.to('cpu')
    assert all(p.device.type == 'cpu' for p in m_q2_cpu.parameters())
    print(f"Q4 ✅  model.to('cpu') moved every Parameter to the CPU")
except (NotImplementedError, TypeError):
    print(f"Q4 ⬜  Module.to(device) recursively moves every registered Parameter and Buffer (checked once AudioMLP is implemented)")

# Q5: Sequential unrolled
seq_q = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
x_q = torch.randn(3, 4)
out_seq = seq_q(x_q)
out_manual = seq_q[2](seq_q[1](seq_q[0](x_q)))
assert torch.allclose(out_seq, out_manual)
print(f"Q5 ✅  Sequential(x) is the same as calling each layer in turn, output shape={tuple(out_seq.shape)}")
print("\n🎉 nn.Module parameter-management whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l61_review = {
    "nn_parameter_understood":  None,  # understand that nn.Parameter puts a tensor into parameters()? True/False
    "audio_mlp_impl":           None,  # AudioMLP implemented correctly, shape/parameter total checks pass? True/False
    "state_dict_understood":    None,  # understand that state_dict holds buffers+parameters, for saving/loading? True/False
    "sequential_forward":       None,  # understand Sequential.forward = calling the submodules in order? True/False
    "whiteboard_passed":        None,  # all 5 whiteboard derivations done by hand? True/False
}

unfilled = [k for k, v in l61_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l61_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L61 fully cleared! On to L62: Dataset and DataLoader')

In [ ]:
# Aurora connection: KVCache's "cache state" corresponds to register_buffer in the torch world
from aurora.llm.kvcache import KVCache

print("aurora.llm.kvcache.KVCache:", KVCache)
print("KVCache base classes:", [b.__name__ for b in KVCache.__bases__])  # pure numpy, doesn't inherit from nn.Module
print("If KVCache were ported to torch, the _k/_v caches should be registered with register_buffer —")
print("  no gradient, moves with .to(device), goes into state_dict: exactly the three traits of a buffer in this lesson's table")

---

→ **Next lesson** · [L62 · `Dataset` and `DataLoader`](L62_kws_dataset.ipynb)

> Next lesson covers **`Dataset` and `DataLoader`**: a custom `__getitem__`, `extract_features` built on Aurora's own mel extractor, and batch-loading keyword audio (falling back automatically to synthetic data when there's no network).